# OpenAI with LangChain

Call OpenAI chat models through LangChain only: `langchain-openai` (`ChatOpenAI`), `langchain-core` prompts/parsers, and LCEL chains.


## 1. Overview

This guide covers:

- What LangChain and LCEL are (short primer)
- Recommended packages: `langchain-openai`, `langchain-core`, `pydantic`
- Loading `.env` and creating `ChatOpenAI` on the **Responses API**
- Building reusable prompts with `ChatPromptTemplate`
- Running LCEL chains: `prompt | llm | StrOutputParser()`
- Structured extraction with `with_structured_output(..., method="json_schema")`
- Streaming with `chain.stream`


## 2. Motivation

Application code usually needs **reusable prompts**, typed outputs, and composable steps — not one-off message dicts. LangChain LCEL (`prompt | llm`) keeps those patterns small and consistent across notebooks.

`ChatOpenAI` still talks to OpenAI under the hood. You configure the model once, then invoke or stream the same chain with different inputs.

## 3. Concepts

### 3.1 Glossary

| Term | Meaning |
|------|--------|
| **`langchain-openai`** | Integration package — import `ChatOpenAI` from here (not deprecated `langchain.chat_models`) |
| **`langchain-core`** | Shared primitives — prompts, parsers, runnables |
| **`ChatOpenAI`** | LangChain chat model for OpenAI |
| **Responses API** | Current OpenAI generation API; enable with `use_responses_api=True` |
| **`ChatPromptTemplate`** | Role-tagged prompt with `{variables}` filled at invoke time |
| **LCEL chain** | Compose with `|`, e.g. `prompt | llm | StrOutputParser()` |
| **`StrOutputParser`** | Turns an `AIMessage` into plain `str` (handles Responses content blocks) |
| **Structured output** | Pydantic schema via `with_structured_output(..., method="json_schema")` |
| **`usage_metadata`** | Token counts on the `AIMessage` (`input_tokens` / `output_tokens`) |

### 3.2 How it works

1. `load_dotenv(root / ".env")` injects `OPENAI_API_KEY` (and optional `OPENAI_MODEL`).
2. Build `ChatOpenAI(model=..., temperature=..., use_responses_api=True)`.
3. Define `ChatPromptTemplate.from_messages([("system", ...), ("human", "...{var}...")])`.
4. Compose `chain = prompt | llm | StrOutputParser()` and call `.invoke` / `.stream`.
5. For typed fields, use `prompt | llm.with_structured_output(Schema, method="json_schema")`.

### 3.3 When to use this pattern

**Use for:** chat prompts, classification, extraction, and any notebook that should reuse the same template.

**Prefer structured output when:** you need enums, booleans, or multi-field JSON you can trust in code.

**Trade-offs:** slightly more abstraction than raw HTTP; much less boilerplate for real apps.


## 4. Architecture

### Setup flow

Run the **next code cell** to render the diagram. Cursor / VS Code Jupyter often strips Mermaid and rich HTML from markdown; `IPython.display.HTML` shows in the cell output instead.


In [1]:
# Architecture diagram -- run this cell to display (IPython HTML)
from IPython.display import HTML, display

ARCH_HTML = """
<div style="font-family:Arial,Helvetica,sans-serif;color:#232F3E;max-width:1000px;">
  <table style="width:100%;border-collapse:separate;border-spacing:10px;table-layout:fixed;">
    <tr>
      <td style="width:25%;vertical-align:top;border:3px solid #7AA116;background:#F2F8E8;padding:10px;">
        <div style="font-size:12px;font-weight:700;color:#5A7A10;text-transform:uppercase;margin-bottom:8px;">1. Packages</div>
        <div style="background:#7AA116;color:#fff;padding:8px;margin:6px 0;text-align:center;font-weight:600;font-size:13px;">langchain-openai<br><span style="font-weight:400;font-size:11px;">ChatOpenAI</span></div>
        <div style="background:#7AA116;color:#fff;padding:8px;margin:6px 0;text-align:center;font-weight:600;font-size:13px;">langchain-core<br><span style="font-weight:400;font-size:11px;">prompts + parsers</span></div>
        <div style="background:#7AA116;color:#fff;padding:8px;margin:6px 0;text-align:center;font-weight:600;font-size:13px;">pydantic<br><span style="font-weight:400;font-size:11px;">schemas</span></div>
      </td>
      <td style="width:25%;vertical-align:top;border:3px solid #FF9900;background:#FFF6E8;padding:10px;">
        <div style="font-size:12px;font-weight:700;color:#D97B00;text-transform:uppercase;margin-bottom:8px;">2. Setup</div>
        <div style="background:#FF9900;color:#232F3E;padding:8px;margin:6px 0;text-align:center;font-weight:600;font-size:13px;">.env<br><span style="font-weight:400;font-size:11px;">API key + model</span></div>
        <div style="background:#FF9900;color:#232F3E;padding:8px;margin:6px 0;text-align:center;font-weight:600;font-size:13px;">ChatOpenAI<br><span style="font-weight:400;font-size:11px;">use_responses_api=True</span></div>
      </td>
      <td style="width:25%;vertical-align:top;border:3px solid #1B6694;background:#E8F3F9;padding:10px;">
        <div style="font-size:12px;font-weight:700;color:#0E4A6B;text-transform:uppercase;margin-bottom:8px;">3. LCEL</div>
        <div style="background:#1B6694;color:#fff;padding:8px;margin:6px 0;text-align:center;font-weight:600;font-size:13px;">ChatPromptTemplate</div>
        <div style="background:#1B6694;color:#fff;padding:8px;margin:6px 0;text-align:center;font-weight:600;font-size:13px;">prompt | llm | StrOutputParser</div>
        <div style="background:#1B6694;color:#fff;padding:8px;margin:6px 0;text-align:center;font-weight:600;font-size:13px;">with_structured_output</div>
      </td>
      <td style="width:25%;vertical-align:top;border:3px solid #8C4FFF;background:#F3ECFF;padding:10px;">
        <div style="font-size:12px;font-weight:700;color:#5A2DBF;text-transform:uppercase;margin-bottom:8px;">4. Outputs</div>
        <div style="background:#8C4FFF;color:#fff;padding:8px;margin:6px 0;text-align:center;font-weight:600;font-size:13px;">str reply</div>
        <div style="background:#8C4FFF;color:#fff;padding:8px;margin:6px 0;text-align:center;font-weight:600;font-size:13px;">Pydantic model</div>
        <div style="background:#8C4FFF;color:#fff;padding:8px;margin:6px 0;text-align:center;font-weight:600;font-size:13px;">usage_metadata</div>
      </td>
    </tr>
  </table>
  <p style="text-align:center;font-weight:700;color:#545B64;">
    .env &rarr; ChatOpenAI &rarr; prompt | llm &rarr; str / Pydantic
  </p>
  <table style="width:100%;border-collapse:collapse;font-size:13px;">
    <tr style="background:#232F3E;color:#fff;">
      <th style="border:1px solid #D5DBDB;padding:8px;text-align:left;">Recipe</th>
      <th style="border:1px solid #D5DBDB;padding:8px;text-align:left;">LCEL shape</th>
      <th style="border:1px solid #D5DBDB;padding:8px;text-align:left;">Returns</th>
    </tr>
    <tr>
      <td style="border:1px solid #D5DBDB;padding:8px;">Plain text</td>
      <td style="border:1px solid #D5DBDB;padding:8px;"><code>prompt | llm | StrOutputParser()</code></td>
      <td style="border:1px solid #D5DBDB;padding:8px;"><code>str</code></td>
    </tr>
    <tr>
      <td style="border:1px solid #D5DBDB;padding:8px;">Structured</td>
      <td style="border:1px solid #D5DBDB;padding:8px;"><code>prompt | llm.with_structured_output(M, method="json_schema")</code></td>
      <td style="border:1px solid #D5DBDB;padding:8px;">Pydantic model</td>
    </tr>
  </table>
</div>
"""

display(HTML(ARCH_HTML))


## 5. What is LangChain?

**LangChain** is a framework for building LLM apps. Instead of hand-rolling message dicts for every call, you work with reusable pieces.

### Recommended packages (current)

| Package | Import from | Use for |
|---------|-------------|---------|
| **`langchain-openai`** | `langchain_openai` | `ChatOpenAI` (OpenAI chat models) |
| **`langchain-core`** | `langchain_core.prompts` / `output_parsers` | `ChatPromptTemplate`, `StrOutputParser`, LCEL runnables |
| **`pydantic`** | `pydantic` | Schemas for `with_structured_output` |
| **`python-dotenv`** | `dotenv` | Load `OPENAI_API_KEY` / `OPENAI_MODEL` from `.env` |

Install (already in this repo's `requirements.txt`):

```bash
pip install -U langchain-openai langchain-core pydantic python-dotenv
```

**Do not** import chat models from deprecated paths such as `langchain.chat_models` or the old completion-only `OpenAI` LLM wrapper for new chat apps. Prefer `ChatOpenAI` from `langchain-openai`.

| Piece | Role |
|-------|------|
| **Chat model** | Talks to a provider (here: `ChatOpenAI` → OpenAI Responses API) |
| **Prompt template** | Fills `{variables}` into system/human messages |
| **Output parser** | Normalizes model output (e.g. `StrOutputParser` → `str`) |
| **Chain** | Wires those pieces together into one callable unit |
| **Structured output** | Forces replies into a Pydantic schema |

### LCEL (LangChain Expression Language)

**LCEL** is the pipe syntax that composes those pieces:

```text
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"question": "..."})
```

- `|` connects steps left → right (prompt output becomes model input)
- `.invoke(inputs)` runs the full chain once
- `.stream(inputs)` yields chunks as tokens arrive
- `.batch([inputs...])` runs many inputs efficiently

OpenAI's current generation surface is the **Responses API**. Set `use_responses_api=True` on `ChatOpenAI` so LangChain calls that API (aligned with `Environment_Setup.ipynb`).


## 6. LangChain Examples

```python
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(root / ".env")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, use_responses_api=True)
chain = prompt | llm | StrOutputParser()
```


In [2]:
# Setup: load .env and create ChatOpenAI (langchain-openai + Responses API)
import os
from pathlib import Path
from typing import Literal  # used later for structured-output enums

from dotenv import load_dotenv  # reads KEY=value from .env into os.environ
from langchain_core.output_parsers import StrOutputParser  # AIMessage -> plain str
from langchain_core.prompts import ChatPromptTemplate  # system/human templates with {vars}
from langchain_openai import ChatOpenAI  # OpenAI chat model integration package
from pydantic import BaseModel, Field  # schemas for with_structured_output

# Notebooks often run with cwd=notebooks/; walk up to the project root.
root = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / ".env").is_file() or (p / "requirements.txt").is_file()  # repo markers
)
load_dotenv(root / ".env")  # explicit path is more reliable than bare load_dotenv()

# Normalize key: strip whitespace and accidental quotes from .env values
api_key = os.getenv("OPENAI_API_KEY", "").strip().strip('"').strip("'")
if not api_key or "your_openai_api_key" in api_key.lower():
    raise SystemExit("Set OPENAI_API_KEY in .env before running the API cells.")
os.environ["OPENAI_API_KEY"] = api_key  # ensure client sees the cleaned value

# Prefer config from .env so the notebook stays portable across machines/models
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini").strip().strip('"').strip("'")

# ChatOpenAI is from langchain-openai (not langchain.chat_models).
# use_responses_api=True -> OpenAI Responses API (current generation surface).
llm = ChatOpenAI(
    model=MODEL,              # which OpenAI model to call
    temperature=0,            # 0 = more deterministic replies
    use_responses_api=True,   # align with Environment_Setup Responses probe
)
print("Model ready:", llm.model_name)
print("API mode  : Responses API (use_responses_api=True)")


Model ready: gpt-4o-mini
API mode  : Responses API (use_responses_api=True)


In [3]:
# Basic LCEL chain: prompt | llm | StrOutputParser
# Builds messages from a template, calls the model, returns plain text.

prompt = ChatPromptTemplate.from_messages(
    [
        # system = stable policy / persona (same for every invoke)
        ("system", "You are a concise SRE assistant. Reply in 1-2 sentences."),
        # human = per-call task; {question} is filled by .invoke({...})
        ("human", "{question}"),
    ]
)

inputs = {"question": "Summarize why health checks should avoid live API calls."}

# One model call: keep AIMessage for usage, then parse to str
msg = (prompt | llm).invoke(inputs)  # returns AIMessage
reply = StrOutputParser().invoke(msg)  # plain str (handles Responses content blocks)

print("Reply:", reply)
print("usage_metadata:", msg.usage_metadata)  # input_tokens / output_tokens / total_tokens


Reply: Health checks should avoid live API calls to prevent unnecessary load on the system and to ensure that the checks are quick and reliable, as they can fail due to transient issues unrelated to the actual health of the service. Instead, using local checks or cached data can provide a more accurate and efficient assessment of system health.
usage_metadata: {'input_tokens': 39, 'output_tokens': 63, 'total_tokens': 102, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}


### 6.1 What is Pydantic?

**Pydantic** is a Python library for data validation using type annotations. You declare the shape of data once; Pydantic parses values, checks types, and raises clear errors when something does not match.

#### BaseModel

BaseModel is the main building block. Subclass it to define fields:

`python
from typing import Literal
from pydantic import BaseModel, Field

class HealthCheckTip(BaseModel):
    summary: str = Field(description="One-sentence summary")
    risk: Literal["low", "medium", "high"]
    avoid_live_calls: bool
`

| Piece | Role |
|-------|------|
| **BaseModel** | Base class for a typed schema object |
| **Field types** | str, ool, int, Literal[...], nested models, etc. |
| **Field(description=...)** | Documents each field (LangChain/OpenAI use this when building the JSON schema) |
| **model_dump()** | Convert a validated instance to a plain dict |

#### How this notebook uses it

1. Define a BaseModel subclass for the answer shape you want.
2. Pass that class to llm.with_structured_output(Schema, method="json_schema").
3. LangChain asks OpenAI to return JSON matching the schema.
4. You get back a **validated Python object** (not a free-text string) — e.g. 	ip.risk is already one of "low" | "medium" | "high".

That is safer than asking the model for "JSON only" and parsing it yourself.


### 6.2 Structured output

Using the Pydantic model from above, ask for a **typed object** instead of free text. OpenAI native JSON-schema mode is selected with method="json_schema".


In [4]:
# Typed extraction via OpenAI json_schema structured outputs
# Define the shape you want; LangChain + OpenAI fill and validate it.

class HealthCheckTip(BaseModel):
    summary: str = Field(description="One-sentence summary")  # free-text field
    risk: Literal["low", "medium", "high"]  # closed enum -- model must pick one
    avoid_live_calls: bool = Field(
        description="Whether live dependency calls should be avoided"
    )


extract_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Extract structured guidance about health-check design. "
            "Use only the schema fields.",  # discourage extra prose
        ),
        # <input> tags delimit untrusted / variable topic text
        ("human", "Topic:\n<input>\n{topic}\n</input>"),
    ]
)

# Bind the schema to the model, then pipe the prompt into that bound model.
# method="json_schema" = OpenAI native structured outputs (preferred).
extract_chain = extract_prompt | llm.with_structured_output(
    HealthCheckTip, method="json_schema"
)

tip = extract_chain.invoke(
    {"topic": "Liveness probes that call a paid third-party API on every check."}
)

print("Parsed:", tip)              # HealthCheckTip instance
print("Dict:", tip.model_dump())   # plain dict for APIs / logging


Parsed: summary='Using liveness probes that call a paid third-party API on every check can lead to unnecessary costs and potential service disruptions.' risk='high' avoid_live_calls=True
Dict: {'summary': 'Using liveness probes that call a paid third-party API on every check can lead to unnecessary costs and potential service disruptions.', 'risk': 'high', 'avoid_live_calls': True}


### 6.3 Streaming

Ask for a **longer** answer and watch tokens arrive over time. The cell below shows:

1. A live-growing reply (one HTML output updated as chunks arrive)
2. A **timestamp log** — wall-clock time and milliseconds since the previous text chunk

Empty Responses API events are skipped so only real text deltas appear in the log.


In [5]:
# Long streaming demo with per-chunk timestamps
# Live reply updates on every text delta; the log records meaningful time gaps.
from datetime import datetime
from html import escape
from time import perf_counter

from IPython.display import HTML, display, update_display

stream_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a clear technical writer. Write in short paragraphs. "
            "Do not use bullet lists.",
        ),
        ("human", "{question}"),
    ]
)

# Slightly higher temperature + longer ask => more tokens over a longer window
stream_llm = ChatOpenAI(
    model=MODEL,
    temperature=0.4,
    use_responses_api=True,
)
stream_chain = stream_prompt | stream_llm | StrOutputParser()

question = (
    "Explain how LangChain LCEL chains work for a beginner. "
    "Cover prompts, the pipe operator, invoke vs stream, and structured output. "
    "Write about 180-220 words so the answer takes a noticeable time to stream."
)


def _render(reply: str, log_lines: list[str]) -> HTML:
    return HTML(
        f"""
<div style="font-family:Arial,Helvetica,sans-serif;color:#232F3E;">
  <div style="font-weight:700;margin-bottom:4px;">Live reply</div>
  <pre style="margin:0 0 12px;padding:10px;background:#F2F3F3;border:1px solid #D5DBDB;
              white-space:pre-wrap;font-family:Consolas,monospace;min-height:6em;">{escape(reply)}</pre>
  <div style="font-weight:700;margin-bottom:4px;">Arrival log (gaps &gt;= 40ms, plus first/last)</div>
  <pre style="margin:0;padding:10px;background:#232F3E;color:#F2F3F3;
              white-space:pre-wrap;font-family:Consolas,monospace;max-height:260px;
              overflow:auto;">{escape(chr(10).join(log_lines))}</pre>
</div>
"""
    )


reply_parts: list[str] = []
log_lines: list[str] = ["time_local       +ms     chars  preview"]
handle = display(_render("(waiting for first tokens...)", log_lines), display_id=True)

t0 = perf_counter()
t_prev = t0
n_text_chunks = 0

for chunk in stream_chain.stream({"question": question}):
    text = str(chunk)  # TextAccessor -> plain str
    if not text:  # skip empty Responses API metadata events
        continue

    now = perf_counter()
    gap_ms = (now - t_prev) * 1000.0
    t_prev = now
    n_text_chunks += 1

    reply_parts.append(text)
    reply = "".join(reply_parts)

    # Log first chunk, last-ish large gaps, and periodic samples so timing is obvious
    should_log = n_text_chunks == 1 or gap_ms >= 40.0 or n_text_chunks % 20 == 0
    if should_log:
        stamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
        preview = text.replace("\\n", "\\\\n")
        if len(preview) > 36:
            preview = preview[:33] + "..."
        log_lines.append(f"{stamp}  {gap_ms:7.1f}  {len(text):5d}  {preview}")

    update_display(_render(reply, log_lines), display_id=handle.display_id)

elapsed_ms = (perf_counter() - t0) * 1000.0
# Always log the final state
stamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
log_lines.append(
    f"{stamp}  ------  -----  done: {elapsed_ms:.0f} ms, {n_text_chunks} text chunks, {len(''.join(reply_parts))} chars"
)
update_display(_render("".join(reply_parts), log_lines), display_id=handle.display_id)


## 7. Implementation notes

1. **Packages** -- `from langchain_openai import ChatOpenAI`; prompts/parsers from `langchain_core`.
2. **Responses API** -- `ChatOpenAI(..., use_responses_api=True)` matches current OpenAI guidance.
3. **`OPENAI_MODEL`** -- Read from `.env`; default `gpt-4o-mini`.
4. **Text chains** -- Prefer `prompt | llm | StrOutputParser()` so `.invoke` returns `str`.
5. **Structured output** -- `with_structured_output(Schema, method="json_schema")` for native JSON schema.
6. **Tokens** -- Read `AIMessage.usage_metadata` (`input_tokens` / `output_tokens`).
7. **Streaming in notebooks** -- Use `display` / `update_display`, skip empty chunks, and `str(chunk)` for Responses API deltas.


## 8. Best practices

- Install provider packages separately (`langchain-openai`), not legacy `langchain.chat_models`.
- Keep keys and model name in `.env`; never hardcode secrets.
- Prefer the Responses API for new OpenAI chat work (`use_responses_api=True`).
- Use `temperature=0` (or low) for extraction; raise slightly for creative drafts.
- Put stable policy in the `system` message; put task data in `human`.
- Delimit untrusted input with tags such as `<input>...</input>`.
- Validate structured results with Pydantic / `Literal`.


## 9. Common failure modes

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| Auth / empty key errors | Missing or placeholder key | Set `OPENAI_API_KEY` in project-root `.env` |
| `ModuleNotFoundError: langchain_openai` | Wrong kernel / missing package | `pip install langchain-openai` in `.venv` |
| Import from `langchain.chat_models` | Deprecated path | `from langchain_openai import ChatOpenAI` |
| List-like `content` when printing | Responses API content blocks | Pipe through `StrOutputParser()` |
| JSON with prose wrapper | Free-text JSON ask | `with_structured_output(..., method="json_schema")` |
| Key works in terminal, not notebook | cwd / `.env` path mismatch | `load_dotenv(root / ".env")` |
| Streaming looks broken / many tiny outputs | `print` per chunk + empty Events | `update_display`; skip empty `str(chunk)` |


## 10. Validation checklist

1. Run setup; confirm `Model ready:` prints and Responses API mode is shown.
2. Run the basic chain; confirm a text reply and `usage_metadata`.
3. Run structured extraction; confirm a `HealthCheckTip` with valid `risk`.
4. Run streaming; confirm tokens print incrementally.


## 11. Summary

- Use **`langchain-openai`** + **`langchain-core`** (not the raw `openai` client in this notebook).
- Enable OpenAI **Responses API** with `use_responses_api=True`.
- Text: `ChatPromptTemplate | ChatOpenAI | StrOutputParser`.
- Structured: `with_structured_output(..., method="json_schema")`.
- This notebook stands alone -- no shared `assets` imports.


